# 8. Robustness Check: Alternative Outcome Cutoffs

This notebook constructs alternative outcome labels with theta = 10, 15, 20, 25 and evaluates all five models from 5_analysis_master.

Outputs at the end include robustness tables versus baseline theta = 20.

In [1]:
import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.sparse import hstack, csr_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
warnings.filterwarnings('ignore')

try:
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.45.0', 'datasets', 'accelerate'])
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    SBERT_AVAILABLE = False

set_seed(SEED)
print('Environment ready. SBERT available:', SBERT_AVAILABLE)

Environment ready. SBERT available: True


In [7]:
def _looks_like_root(p: Path) -> bool:
    required = [
        Path('cleaned_data/us_notif_clean_wide.csv'),
        Path('clean_out/hs4_imports_with_notif_exposure.csv'),
        Path('clean_out/notif_hs4_detailed.csv'),
    ]
    return p.exists() and all((p / rel).exists() for rel in required)

def _candidate_roots():
    seen = set()

    def _add(path_like):
        if not path_like:
            return
        p = Path(path_like).expanduser()
        key = str(p)
        if key not in seen:
            seen.add(key)
            yield p

    env_root = os.getenv('PROJECT_ROOT')
    if env_root:
        yield from _add(env_root)

    # VS Code/Jupyter may expose the notebook path even when cwd is different.
    nb_path = globals().get('__vsc_ipynb_file__') or os.getenv('JPY_SESSION_NAME')
    if nb_path:
        nb_parent = Path(nb_path).expanduser().resolve().parent
        yield from _add(nb_parent)
        for parent in nb_parent.parents:
            yield from _add(parent)

    cwd = Path.cwd()
    yield from _add(cwd)
    for parent in cwd.parents:
        yield from _add(parent)

    hardcoded = [
        Path(r'G:/My Drive/Project'),
        Path(r'g:/My Drive/Project'),
        Path('/mnt/g/My Drive/Project'),
        Path('/content/drive/MyDrive/Project'),
        Path('/content/drive/My Drive/Project'),
    ]
    for p in hardcoded:
        yield from _add(p)

def _resolve_project_root():
    tried = []
    for cand in _candidate_roots():
        tried.append(str(cand))
        if _looks_like_root(cand):
            return cand, tried

    # If running in Colab, try mounting Drive once and re-check common paths.
    if Path('/content').exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
            for cand in [Path('/content/drive/MyDrive/Project'), Path('/content/drive/My Drive/Project')]:
                tried.append(str(cand))
                if _looks_like_root(cand):
                    return cand, tried
        except Exception:
            pass

    return None, tried

PROJECT_ROOT, _TRIED_ROOTS = _resolve_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate project root. Set env var PROJECT_ROOT to your folder or switch to a kernel that can access your workspace. '
        f'cwd={Path.cwd()} | tried={_TRIED_ROOTS[:16]}'
    )

CLEAN = PROJECT_ROOT / 'clean_out'
CLEANED = PROJECT_ROOT / 'cleaned_data'

NOTIFS_PATH = CLEANED / 'us_notif_clean_wide.csv'
PANEL_PATH = CLEAN / 'hs4_imports_with_notif_exposure.csv'
NOTIF_HS4_PATH = CLEAN / 'notif_hs4_detailed.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Files found:', NOTIFS_PATH.exists(), PANEL_PATH.exists(), NOTIF_HS4_PATH.exists())

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/Project
Files found: True True True


In [3]:
# Config
K = 6
THETA_LIST = [0.10, 0.15, 0.20, 0.25]
TOP_FRAC = 0.10

PRETRAIN_END = pd.Timestamp('2018-12-31')
BRIDGE_START = pd.Timestamp('2019-01-01')
BRIDGE_END = pd.Timestamp('2019-12-31')
FINETUNE_START = pd.Timestamp('2020-01-01')
FINETUNE_END = pd.Timestamp('2023-12-31')
TEST_START = pd.Timestamp('2024-01-01')
TEST_END = pd.Timestamp('2025-12-31')

FAST_MODE = True
DISTIL_MODEL = 'distilbert-base-uncased'
DISTIL_MAX_LEN = 256
DISTIL_EPOCHS = 1 if FAST_MODE else 3

def precision_recall_at_top_frac(y_true, y_score, frac=0.10):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    n = len(y_true)
    k = max(1, int(np.ceil(frac * n)))
    idx = np.argsort(-y_score)[:k]
    y_top = y_true[idx]
    return float(y_top.mean()), float(y_top.sum() / max(1, y_true.sum())), k

def safe_auc(y, p):
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

In [8]:
# Load static inputs
panel = pd.read_csv(PANEL_PATH, dtype={'country': str, 'hs4': str})
panel['date'] = pd.to_datetime(panel['date'], errors='coerce')
panel = panel.dropna(subset=['date']).copy()

notif_hs4 = pd.read_csv(NOTIF_HS4_PATH, dtype={'hs4': str})
notif_hs4['t0'] = pd.to_datetime(
    notif_hs4['year'].astype(str) + '-' + notif_hs4['month'].astype(str).str.zfill(2) + '-01',
    errors='coerce'
)
notif_hs4 = notif_hs4.dropna(subset=['t0', 'notified_document', 'hs4']).copy()
notif_hs4['w'] = pd.to_numeric(notif_hs4['w'], errors='coerce').fillna(0)
den = notif_hs4.groupby('notified_document')['w'].transform('sum')
notif_hs4 = notif_hs4[den > 0].copy()
notif_hs4['w'] = notif_hs4['w'] / den

notifs = pd.read_csv(NOTIFS_PATH, dtype=str)
for txt_col in ['full_text', 'text', 'description', 'title']:
    if txt_col in notifs.columns:
        TEXTCOL = txt_col
        break
else:
    raise ValueError('No text column found.')

notifs['notified_document'] = notifs['notified_document'].fillna('').astype(str).str.strip()
notifs['distribution_date'] = pd.to_datetime(notifs['distribution_date'], errors='coerce')
notifs[TEXTCOL] = notifs[TEXTCOL].fillna('').astype(str).str.strip()
notifs = notifs[(notifs['notified_document'] != '') & (~notifs['distribution_date'].isna())].copy()
notifs['text_len'] = notifs[TEXTCOL].str.len()
notifs_text = (
    notifs.sort_values(['notified_document', 'text_len'], ascending=[True, False])
          .drop_duplicates(subset=['notified_document'], keep='first')
          [['notified_document', 'distribution_date', TEXTCOL]]
          .rename(columns={TEXTCOL: 'text'})
)

print('Loaded panel:', len(panel))
print('Loaded notif_hs4:', len(notif_hs4))
print('Loaded notifs_text:', len(notifs_text))

Loaded panel: 522855
Loaded notif_hs4: 28384
Loaded notifs_text: 5144


In [9]:
def add_tau(d):
    d = d.copy()
    d['tau'] = (d['date'].dt.to_period('M') - d['notif_month'].dt.to_period('M')).apply(lambda x: x.n)
    return d

def build_labels(theta):
    thr = np.log(1 - theta)

    notif_map = notif_hs4[['notified_document', 'hs4', 't0', 'w']].copy()
    notif_map = notif_map.rename(columns={'t0': 'notif_month'})

    cp = panel[['country', 'hs4', 'date', 'import_value']].merge(notif_map, on='hs4', how='inner')
    cp = add_tau(cp)
    cp = cp[cp['tau'].between(-3, K)].copy()
    cp['w_imp'] = cp['w'] * cp['import_value']

    cp_agg = cp.groupby(['notified_document', 'country', 'hs4', 'tau'], as_index=False).agg(imp=('w_imp', 'sum'))
    cp_base = cp_agg[cp_agg['tau'].between(-3, -1)].groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean()
    cp_post = cp_agg[cp_agg['tau'].between(1, K)].groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean()
    cp_wide = cp_base.merge(cp_post, on=['notified_document', 'country', 'hs4'], suffixes=('_base', '_post'), how='inner')
    cp_wide['delta_cp'] = np.log1p(cp_wide['imp_post']) - np.log1p(cp_wide['imp_base'])

    hs4_tot = panel.groupby(['hs4', 'date'], as_index=False)['import_value'].sum().rename(columns={'import_value': 'imp_hs4'})
    hs4_tot = hs4_tot.merge(notif_map[['notified_document', 'hs4', 'notif_month', 'w']], on='hs4', how='inner')
    hs4_tot = add_tau(hs4_tot)
    hs4_tot = hs4_tot[hs4_tot['tau'].between(-3, K)].copy()
    hs4_tot['w_imp'] = hs4_tot['w'] * hs4_tot['imp_hs4']

    hs4_agg = hs4_tot.groupby(['notified_document', 'hs4', 'tau'], as_index=False).agg(imp=('w_imp', 'sum'))
    hs4_base = hs4_agg[hs4_agg['tau'].between(-3, -1)].groupby(['notified_document', 'hs4'], as_index=False)['imp'].mean()
    hs4_post = hs4_agg[hs4_agg['tau'].between(1, K)].groupby(['notified_document', 'hs4'], as_index=False)['imp'].mean()
    hs4_wide = hs4_base.merge(hs4_post, on=['notified_document', 'hs4'], suffixes=('_base', '_post'), how='inner')
    hs4_wide['delta_hs4'] = np.log1p(hs4_wide['imp_post']) - np.log1p(hs4_wide['imp_base'])

    exp_tot = panel.groupby(['country', 'date'], as_index=False)['import_value'].sum().rename(columns={'import_value': 'imp_exp'})
    exp_keys = cp[['notified_document', 'country', 'hs4', 'notif_month', 'w']].drop_duplicates()
    exp_tot = exp_tot.merge(exp_keys, on='country', how='inner')
    exp_tot = add_tau(exp_tot)
    exp_tot = exp_tot[exp_tot['tau'].between(-3, K)].copy()
    exp_tot['w_imp'] = exp_tot['w'] * exp_tot['imp_exp']

    exp_agg = exp_tot.groupby(['notified_document', 'country', 'hs4', 'tau'], as_index=False).agg(imp=('w_imp', 'sum'))
    exp_base = exp_agg[exp_agg['tau'].between(-3, -1)].groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean()
    exp_post = exp_agg[exp_agg['tau'].between(1, K)].groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean()
    exp_wide = exp_base.merge(exp_post, on=['notified_document', 'country', 'hs4'], suffixes=('_base', '_post'), how='inner')
    exp_wide['delta_exp'] = np.log1p(exp_wide['imp_post']) - np.log1p(exp_wide['imp_base'])

    ev = cp_wide.merge(hs4_wide[['notified_document', 'hs4', 'delta_hs4']], on=['notified_document', 'hs4'], how='left')\
                .merge(exp_wide[['notified_document', 'country', 'hs4', 'delta_exp']], on=['notified_document', 'country', 'hs4'], how='left')

    ev['delta_vs_hs4'] = ev['delta_cp'] - ev['delta_hs4']
    ev['delta_vs_exp'] = ev['delta_cp'] - ev['delta_exp']

    ycol = f'Y_struct_abn_t{int(theta*100):02d}'
    ev[ycol] = ((ev['delta_vs_hs4'] <= thr) & (ev['delta_vs_exp'] <= thr)).astype(int)

    labels = ev[['notified_document', 'country', 'hs4', ycol]].copy()
    return labels, ycol

def build_modeling_df(labels, ycol):
    df = labels.merge(notifs_text, on='notified_document', how='inner')
    df = df[['notified_document', 'country', 'hs4', 'distribution_date', 'text', ycol]].copy()
    df = df.dropna(subset=['distribution_date', ycol]).copy()
    df[ycol] = df[ycol].astype(int)
    df['text'] = df['text'].fillna('').astype(str)
    return df.sort_values('distribution_date').reset_index(drop=True)

In [10]:
def temporal_split(df, ycol):
    notif_dates = df[['notified_document', 'distribution_date']].drop_duplicates()

    pretrain_notifs = set(notif_dates.loc[notif_dates['distribution_date'] <= PRETRAIN_END, 'notified_document'])
    bridge_notifs = set(notif_dates.loc[(notif_dates['distribution_date'] >= BRIDGE_START) & (notif_dates['distribution_date'] <= BRIDGE_END), 'notified_document'])
    finetune_notifs = set(notif_dates.loc[(notif_dates['distribution_date'] >= FINETUNE_START) & (notif_dates['distribution_date'] <= FINETUNE_END), 'notified_document'])
    test_notifs = set(notif_dates.loc[(notif_dates['distribution_date'] >= TEST_START) & (notif_dates['distribution_date'] <= TEST_END), 'notified_document'])

    df_pre = df[df['notified_document'].isin(pretrain_notifs)].copy()
    df_br = df[df['notified_document'].isin(bridge_notifs)].copy()
    df_tr = df[df['notified_document'].isin(finetune_notifs)].copy()
    df_te = df[df['notified_document'].isin(test_notifs)].copy()

    ft_notif_dates = df_tr[['notified_document', 'distribution_date']].drop_duplicates().sort_values('distribution_date')
    n_val = max(1, int(np.ceil(0.10 * len(ft_notif_dates)))) if len(ft_notif_dates) > 1 else 0
    val_notifs = set(ft_notif_dates['notified_document'].tail(n_val)) if n_val > 0 else set()
    tr_notifs = set(ft_notif_dates['notified_document']) - val_notifs

    df_dtr = df_tr[df_tr['notified_document'].isin(tr_notifs)].copy()
    df_dval = df_tr[df_tr['notified_document'].isin(val_notifs)].copy()
    if len(df_dval) == 0 and len(df_dtr) > 0:
        df_dval = df_dtr.sample(min(1000, len(df_dtr)), random_state=SEED).copy()

    return df_pre, df_br, df_tr, df_te, df_dtr, df_dval

def eval_metrics(name, y_true, p):
    prec, rec, _ = precision_recall_at_top_frac(y_true, p, TOP_FRAC)
    return {
        'model': name,
        'roc_auc': safe_auc(y_true, p),
        'avg_precision': average_precision_score(y_true, p),
        'precision_top10pct': prec,
        'recall_top10pct': rec
    }

def run_baselines(df_tr, df_te, ycol):
    groups = df_tr['notified_document'].values
    cv = GroupKFold(n_splits=5)

    country_pipe = Pipeline([
        ('prep', ColumnTransformer([('country', OneHotEncoder(handle_unknown='ignore'), ['country'])], remainder='drop')),
        ('clf', LogisticRegression(penalty='l2', solver='lbfgs', class_weight='balanced', max_iter=5000, random_state=SEED))
    ])
    country_pipe.fit(df_tr[['country']], df_tr[ycol])
    p_country = country_pipe.predict_proba(df_te[['country']])[:, 1]

    text_pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, stop_words='english', max_features=200000)),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', class_weight='balanced', max_iter=5000, random_state=SEED))
    ])
    gs_text = GridSearchCV(text_pipe, {'clf__C': [0.5, 1.0], 'clf__l1_ratio': [0.25, 0.5]}, scoring='average_precision', cv=cv.split(df_tr, df_tr[ycol], groups), n_jobs=-1, verbose=0)
    gs_text.fit(df_tr['text'], df_tr[ycol])
    p_text = gs_text.best_estimator_.predict_proba(df_te['text'])[:, 1]

    tc_pipe = Pipeline([
        ('prep', ColumnTransformer([
            ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, stop_words='english', max_features=200000), 'text'),
            ('country', OneHotEncoder(handle_unknown='ignore'), ['country'])
        ], remainder='drop', sparse_threshold=0.3)),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', class_weight='balanced', max_iter=5000, random_state=SEED))
    ])
    gs_tc = GridSearchCV(tc_pipe, {'clf__C': [0.5, 1.0], 'clf__l1_ratio': [0.0, 0.25, 0.5]}, scoring='average_precision', cv=cv.split(df_tr, df_tr[ycol], groups), n_jobs=-1, verbose=0)
    gs_tc.fit(df_tr[['text', 'country']], df_tr[ycol])
    p_tc = gs_tc.best_estimator_.predict_proba(df_te[['text', 'country']])[:, 1]

    y = df_te[ycol].values
    out = [
        eval_metrics('Country-only', y, p_country),
        eval_metrics('Text-only (TF-IDF)', y, p_text),
        eval_metrics('Text + Country (TF-IDF)', y, p_tc)
    ]

    if SBERT_AVAILABLE:
        st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        emb_tr = st_model.encode(df_tr['text'].fillna('').tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        emb_te = st_model.encode(df_te['text'].fillna('').tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)

        enc = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        c_tr = enc.fit_transform(df_tr[['country']])
        c_te = enc.transform(df_te[['country']])

        Xtr = hstack([csr_matrix(emb_tr), c_tr])
        Xte = hstack([csr_matrix(emb_te), c_te])

        emb_clf = LogisticRegression(penalty='l2', solver='lbfgs', class_weight='balanced', max_iter=5000, random_state=SEED)
        gs_emb = GridSearchCV(emb_clf, {'C': [0.5, 1.0]}, scoring='average_precision', cv=cv.split(Xtr, df_tr[ycol], groups), n_jobs=-1, verbose=0)
        gs_emb.fit(Xtr, df_tr[ycol])
        p_emb = gs_emb.best_estimator_.predict_proba(Xte)[:, 1]
        out.append(eval_metrics('Embeddings + Country (SBERT)', y, p_emb))

    return out

def run_distilbert(df_dtr, df_dval, df_te, ycol, theta_tag):
    tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)

    for fr in (df_dtr, df_dval, df_te):
        fr['model_text'] = ('country: ' + fr['country'].astype(str) + ' ; hs4: ' + fr['hs4'].astype(str) + ' ; text: ' + fr['text'].astype(str)).str.strip()

    def to_ds(frame):
        ds = Dataset.from_pandas(frame[['model_text', ycol]].rename(columns={ycol: 'labels'}), preserve_index=False)
        ds = ds.map(lambda b: tokenizer(b['model_text'], padding='max_length', truncation=True, max_length=DISTIL_MAX_LEN), batched=True)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
        return ds

    tr_ds = to_ds(df_dtr)
    va_ds = to_ds(df_dval if len(df_dval) > 0 else df_dtr.sample(min(1000, len(df_dtr)), random_state=SEED))
    te_ds = to_ds(df_te)

    model = AutoModelForSequenceClassification.from_pretrained(DISTIL_MODEL, num_labels=2)
    args = TrainingArguments(
        output_dir=str(CLEAN / 'distilbert_run_robust' / f'theta_{theta_tag}'),
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        num_train_epochs=DISTIL_EPOCHS,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        fp16=torch.cuda.is_available(),
        save_total_limit=1,
        report_to='none',
        logging_steps=50,
        seed=SEED
    )

    trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds, tokenizer=tokenizer, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.train()

    pred = trainer.predict(te_ds)
    y = pred.label_ids.astype(int)
    p = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()

    out = eval_metrics('DistilBERT', y, p)

    del trainer, model, tr_ds, va_ds, te_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

In [11]:
# Main robustness loop
all_rows = []
label_rows = []

for theta in THETA_LIST:
    theta_tag = f'{int(theta*100):02d}'
    print('\n' + '='*80)
    print(f'Theta = {theta:.2f}')
    print('='*80)

    labels, ycol = build_labels(theta)
    df = build_modeling_df(labels, ycol)

    label_rows.append({'theta': theta, 'ycol': ycol, 'n_rows': len(df), 'pos_rate': float(df[ycol].mean()) if len(df) else np.nan})
    print(f'Rows: {len(df):,} | pos_rate: {df[ycol].mean():.4f}')

    df_pre, df_br, df_tr, df_te, df_dtr, df_dval = temporal_split(df, ycol)
    if len(df_tr) == 0 or len(df_te) == 0:
        print('Skipping: empty train/test split.')
        continue

    base_results = run_baselines(df_tr, df_te, ycol)
    for r in base_results:
        r['theta'] = theta
        all_rows.append(r)

    dres = run_distilbert(df_dtr, df_dval, df_te, ycol, theta_tag)
    dres['theta'] = theta
    all_rows.append(dres)

metrics_theta = pd.DataFrame(all_rows)
labels_theta = pd.DataFrame(label_rows)

display(labels_theta)
display(metrics_theta.sort_values(['theta', 'avg_precision'], ascending=[True, False]))


Theta = 0.10
Rows: 52,033 | pos_rate: 0.2490


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/10404 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.422700,0.411835


,theta,ycol,n_rows,pos_rate
0,0.10,Y_struct_abn_t10,52033,0.249015
1,0.15,Y_struct_abn_t15,52033,0.212788
2,0.20,Y_struct_abn_t20,52033,0.181443
3,0.25,Y_struct_abn_t25,52033,0.156766


,model,roc_auc,avg_precision,precision_top10pct,recall_top10pct,theta
4,DistilBERT,0.549973,0.285165,0.293948,0.117331,0.10
0,Country-only,0.568716,0.283133,0.292988,0.116948,0.10
2,Text + Country (TF-IDF),0.545884,0.277830,0.292988,0.116948,0.10
3,Embeddings + Country (SBERT),0.532977,0.274204,0.299712,0.119632,0.10
1,Text-only (TF-IDF),0.496673,0.250354,0.254563,0.101610,0.10
9,DistilBERT,0.550992,0.246367,0.268972,0.125956,0.15
7,Text + Country (TF-IDF),0.552075,0.244012,0.258405,0.121008,0.15
5,Country-only,0.562682,0.242234,0.265130,0.124157,0.15
8,Embeddings + Country (SBERT),0.535214,0.238994,0.260327,0.121907,0.15
6,Text-only (TF-IDF),0.494670,0.213039,0.208453,0.097616,0.15


In [12]:
# Robustness tables vs baseline theta=0.20
metrics_theta = metrics_theta.sort_values(['theta', 'model']).reset_index(drop=True)
baseline = metrics_theta[metrics_theta['theta'] == 0.20][['model', 'roc_auc', 'avg_precision', 'precision_top10pct', 'recall_top10pct']].copy()
baseline = baseline.rename(columns={
    'roc_auc': 'roc_auc_base20',
    'avg_precision': 'avg_precision_base20',
    'precision_top10pct': 'precision_top10pct_base20',
    'recall_top10pct': 'recall_top10pct_base20'
})

compare = metrics_theta.merge(baseline, on='model', how='left')
for m in ['roc_auc', 'avg_precision', 'precision_top10pct', 'recall_top10pct']:
    compare[f'delta_{m}_vs20'] = compare[m] - compare[f'{m}_base20']

best_by_theta = metrics_theta.sort_values(['theta', 'avg_precision'], ascending=[True, False]).groupby('theta', as_index=False).first()

out1 = CLEAN / 'robustness_model_metrics_by_theta.csv'
out2 = CLEAN / 'robustness_compare_vs_theta20.csv'
out3 = CLEAN / 'robustness_best_model_by_theta.csv'
out4 = CLEAN / 'robustness_label_summary_by_theta.csv'

metrics_theta.to_csv(out1, index=False)
compare.to_csv(out2, index=False)
best_by_theta.to_csv(out3, index=False)
labels_theta.to_csv(out4, index=False)

print('Saved robustness outputs:')
print(' -', out1)
print(' -', out2)
print(' -', out3)
print(' -', out4)

display(compare.sort_values(['model', 'theta']))

Saved robustness outputs:
 - /content/drive/MyDrive/Project/clean_out/robustness_model_metrics_by_theta.csv
 - /content/drive/MyDrive/Project/clean_out/robustness_compare_vs_theta20.csv
 - /content/drive/MyDrive/Project/clean_out/robustness_best_model_by_theta.csv
 - /content/drive/MyDrive/Project/clean_out/robustness_label_summary_by_theta.csv


,model,roc_auc,avg_precision,precision_top10pct,recall_top10pct,theta,roc_auc_base20,avg_precision_base20,precision_top10pct_base20,recall_top10pct_base20,delta_roc_auc_vs20,delta_avg_precision_vs20,delta_precision_top10pct_vs20,delta_recall_top10pct_vs20
0,Country-only,0.568716,0.283133,0.292988,0.116948,0.10,0.578511,0.213848,0.243996,0.135974,-0.009794,0.069286,0.048991,-0.019026
5,Country-only,0.562682,0.242234,0.265130,0.124157,0.15,0.578511,0.213848,0.243996,0.135974,-0.015828,0.028386,0.021134,-0.011818
10,Country-only,0.578511,0.213848,0.243996,0.135974,0.20,0.578511,0.213848,0.243996,0.135974,0.000000,0.000000,0.000000,0.000000
15,Country-only,0.579819,0.185862,0.210375,0.136364,0.25,0.578511,0.213848,0.243996,0.135974,0.001309,-0.027985,-0.033622,0.000389
1,DistilBERT,0.549973,0.285165,0.293948,0.117331,0.10,0.561591,0.207850,0.226705,0.126338,-0.011619,0.077315,0.067243,-0.009007
6,DistilBERT,0.550992,0.246367,0.268972,0.125956,0.15,0.561591,0.207850,0.226705,0.126338,-0.010600,0.038517,0.042267,-0.000382
11,DistilBERT,0.561591,0.207850,0.226705,0.126338,0.20,0.561591,0.207850,0.226705,0.126338,0.000000,0.000000,0.000000,0.000000
16,DistilBERT,0.564597,0.184927,0.210375,0.136364,0.25,0.561591,0.207850,0.226705,0.126338,0.003006,-0.022924,-0.016330,0.010025
2,Embeddings + Country (SBERT),0.532977,0.274204,0.299712,0.119632,0.10,0.557010,0.211769,0.245917,0.137045,-0.024033,0.062436,0.053794,-0.017413
7,Embeddings + Country (SBERT),0.535214,0.238994,0.260327,0.121907,0.15,0.557010,0.211769,0.245917,0.137045,-0.021796,0.027225,0.014409,-0.015138


In [13]:
# Compact comparison tables
pivot_ap = metrics_theta.pivot(index='model', columns='theta', values='avg_precision').round(4)
pivot_auc = metrics_theta.pivot(index='model', columns='theta', values='roc_auc').round(4)
pivot_p10 = metrics_theta.pivot(index='model', columns='theta', values='precision_top10pct').round(4)

display(pivot_ap)
display(pivot_auc)
display(pivot_p10)

pivot_ap.to_csv(CLEAN / 'robustness_table_ap_by_theta.csv')
pivot_auc.to_csv(CLEAN / 'robustness_table_auc_by_theta.csv')
pivot_p10.to_csv(CLEAN / 'robustness_table_precision_top10_by_theta.csv')

print('Exported compact tables to clean_out/.')

theta,0.10,0.15,0.20,0.25
model,,,,
Country-only,0.2831,0.2422,0.2138,0.1859
DistilBERT,0.2852,0.2464,0.2079,0.1849
Embeddings + Country (SBERT),0.2742,0.2390,0.2118,0.1853
Text + Country (TF-IDF),0.2778,0.2440,0.2135,0.1896
Text-only (TF-IDF),0.2504,0.2130,0.1776,0.1542


theta,0.10,0.15,0.20,0.25
model,,,,
Country-only,0.5687,0.5627,0.5785,0.5798
DistilBERT,0.5500,0.5510,0.5616,0.5646
Embeddings + Country (SBERT),0.5330,0.5352,0.5570,0.5634
Text + Country (TF-IDF),0.5459,0.5521,0.5628,0.5715
Text-only (TF-IDF),0.4967,0.4947,0.4970,0.4977


theta,0.10,0.15,0.20,0.25
model,,,,
Country-only,0.2930,0.2651,0.2440,0.2104
DistilBERT,0.2939,0.2690,0.2267,0.2104
Embeddings + Country (SBERT),0.2997,0.2603,0.2459,0.2171
Text + Country (TF-IDF),0.2930,0.2584,0.2305,0.2075
Text-only (TF-IDF),0.2546,0.2085,0.1758,0.1604


Exported compact tables to clean_out/.


## End

Primary outputs:
- clean_out/robustness_model_metrics_by_theta.csv
- clean_out/robustness_compare_vs_theta20.csv
- clean_out/robustness_best_model_by_theta.csv
- clean_out/robustness_table_ap_by_theta.csv
- clean_out/robustness_table_auc_by_theta.csv
- clean_out/robustness_table_precision_top10_by_theta.csv

Set FAST_MODE=False in the config cell for a slower but more exhaustive run.